<a href="https://colab.research.google.com/github/SatoshiTomita/Algorithm/blob/main/MNIST10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''MobileNetV2 in PyTorch.

See the paper "Inverted Residuals and Linear Bottlenecks:
Mobile Networks for Classification, Detection and Segmentation" for more details.
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    '''expand + depthwise + pointwise'''
    def __init__(self, in_planes, out_planes, expansion, stride):
        super(Block, self).__init__()
        self.stride = stride

        planes = expansion * in_planes
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, groups=planes, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn3 = nn.BatchNorm2d(out_planes)

        self.shortcut = nn.Sequential()
        if stride == 1 and in_planes != out_planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = out + self.shortcut(x) if self.stride==1 else out
        return out


class MobileNetV2(nn.Module):
    cfg = [(1,  16, 1, 1),
           (4,  24, 2, 1),
           (4,  32, 3, 2),
           (4,  64, 4, 2),
           (4,  96, 3, 1),
           (4, 160, 3, 2),
           (4, 320, 1, 1)]

    def __init__(self, num_classes=10):
        super(MobileNetV2, self).__init__()
        # 全体的なチャンネル幅を 0.5倍 (alpha=0.5) に設定
        alpha = 0.5
        c1 = int(32 * alpha)

        self.conv1 = nn.Conv2d(3, c1, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c1)

        # _make_layers 内でも alpha を適用するように修正が必要
        self.layers = self._make_layers(in_planes=c1, alpha=alpha)

        # 最終層を 1280 -> 256 に大幅カット
        last_channels = 256
        self.conv2 = nn.Conv2d(int(320 * alpha), last_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(last_channels)
        self.linear = nn.Linear(last_channels, num_classes)

    def _make_layers(self, in_planes, alpha):
        layers = []
        for expansion, out_planes, num_blocks, stride in self.cfg:
            # 各層の出力チャンネルに alpha を掛ける
            out_planes = int(out_planes * alpha)
            strides = [stride] + [1]*(num_blocks-1)
            for stride in strides:
                layers.append(Block(in_planes, out_planes, expansion, stride))
                in_planes = out_planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layers(out)
        out = F.relu(self.bn2(self.conv2(out)))
        # GAPを用いた実装
        out = F.adaptive_avg_pool2d(out, 1)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def test():
    net = MobileNetV2()
    x = torch.randn(2,3,32,32)
    y = net(x)
    print(y.size())

In [ ]:
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as transforms

import os
import argparse
import sys
import time

TOTAL_BAR_LENGTH = 30
_last_time = time.time()
_begin_time = _last_time
train_losses = []
train_accs = []
test_losses = []
test_accs = []

def progress_bar(current: int, total: int, msg: str | None = None) -> None:
    global _last_time, _begin_time

    if current == 0:
        _begin_time = time.time()
        _last_time = _begin_time

    cur_len = int(TOTAL_BAR_LENGTH * current / total)
    rest_len = TOTAL_BAR_LENGTH - cur_len - 1

    bar = "[" + "=" * cur_len + ">" + "." * rest_len + "]"

    now = time.time()
    step_time = now - _last_time
    _last_time = now
    total_time = now - _begin_time

    info = f" Step: {format_time(step_time)} | Tot: {format_time(total_time)}"
    if msg:
        info += f" | {msg}"

    sys.stdout.write(f"\r{bar}{info} {current + 1}/{total}")
    if current == total - 1:
        sys.stdout.write("\n")
    sys.stdout.flush()


def format_time(seconds: float) -> str:
    for unit, scale in [("s", 1), ("ms", 1e-3)]:
        if seconds >= scale:
            return f"{int(seconds / scale)}{unit}"
    return "0ms"

parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
parser.add_argument('--resume', '-r', action='store_true',
                    help='resume from checkpoint')
args, _ = parser.parse_known_args()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
best_acc = 0
start_epoch = 0

# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=256, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=256, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print('==> Building model..')
# MobileNetを使用
net = MobileNetV2()
net = net.to(device)
total_params = sum(p.numel() for p in net.parameters())
print(f"Total Parameters: {total_params}")
if device == 'cuda':
    net = torch.nn.DataParallel(net)
    cudnn.benchmark = True

if args.resume:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(net.parameters(), lr=args.lr,
                      momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)


# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        progress_bar(
            batch_idx, len(trainloader),
            'Loss: %.3f | Acc: %.3f%% (%d/%d)'
            % (train_loss/(batch_idx+1), 100.*correct/total, correct, total)
        )

    avg_loss = train_loss / len(trainloader)
    acc = 100. * correct / total
    return avg_loss, acc

def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)

            outputs1 = net(inputs)
            inputs_flipped = torch.flip(inputs, [3])
            outputs2 = net(inputs_flipped)
            outputs = (outputs1 + outputs2) / 2

            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar(
                batch_idx, len(testloader),
                'Loss: %.3f | Acc: %.3f%% (%d/%d)'
                % (test_loss/(batch_idx+1), 100.*correct/total, correct, total)
            )

    avg_loss = test_loss / len(testloader)
    acc = 100. * correct / total

    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        os.makedirs('checkpoint', exist_ok=True)
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

    return avg_loss, acc


for epoch in range(start_epoch, start_epoch+50):
    tr_loss, tr_acc = train(epoch)
    te_loss, te_acc = test(epoch)

    train_losses.append(tr_loss)
    train_accs.append(tr_acc)
    test_losses.append(te_loss)
    test_accs.append(te_acc)

    scheduler.step()

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label='Train Loss')
plt.plot(epochs, test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, train_accs, label='Train Acc')
plt.plot(epochs, test_accs, label='Test Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.show()

==> Preparing data..


100%|██████████| 170M/170M [00:01<00:00, 107MB/s]


==> Building model..
Total Parameters: 378162

Epoch: 0
[=============================>] Step: 904ms | Tot: 792s | Loss: 1.840 | Acc: 35.952% (17976/50000) 196/196
[=============================>] Step: 155ms | Tot: 62s | Loss: 1.650 | Acc: 46.010% (4601/10000) 40/40
Saving..

Epoch: 1
[=============================>] Step: 976ms | Tot: 775s | Loss: 1.501 | Acc: 53.644% (26822/50000) 196/196
[=============================>] Step: 118ms | Tot: 64s | Loss: 1.538 | Acc: 53.140% (5314/10000) 40/40
Saving..

Epoch: 2
[=============================>] Step: 1s | Tot: 786s | Loss: 1.315 | Acc: 63.188% (31594/50000) 196/196
[=============================>] Step: 110ms | Tot: 85s | Loss: 1.246 | Acc: 66.970% (6697/10000) 40/40
Saving..

Epoch: 3
[=============================>] Step: 1s | Tot: 783s | Loss: 1.200 | Acc: 68.702% (34351/50000) 196/196
[=============================>] Step: 131ms | Tot: 66s | Loss: 1.192 | Acc: 69.540% (6954/10000) 40/40
Saving..

Epoch: 4
[====>....................